# B3. In-sample R² of every equation on every Pareto front

B3: exact in-sample R2 for every equation on the Pareto fronts used in the paper,
evaluated on the rows the search actually trained on (all selected features + target
present). Fixes the '1 - loss/var' approximation used earlier for the units front.
Output: Paper/figures/data/front_r2_recomputed.csv

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()          # notebooks/ folder
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import json, re, sys
from pathlib import Path
import numpy as np, pandas as pd, sympy as sp
from sklearn.metrics import r2_score
from termmap_utils import DATA, equation_variables, load_front
from style import *   # PM25, O3, NAVY, GREY, RED, LIGHT, ROOT, DATA_OUT, save_fig, panel_label

In [2]:
# Analysis
def evaluate(eq, d, target):
    vs = sorted(equation_variables(eq))
    sub = d
    ns = {v: sp.Symbol(v) for v in vs}
    expr = sp.sympify(eq, locals={**ns, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    f = sp.lambdify([ns[v] for v in vs], expr, "numpy")
    with np.errstate(all="ignore"):
        pred = f(*[sub[v].to_numpy(float) for v in vs]) if vs else np.full(len(sub), float(expr))
    pred = np.broadcast_to(np.asarray(pred, float), (len(sub),))
    ok = np.isfinite(pred)
    return r2_score(sub[target].to_numpy(float)[ok], pred[ok]), int(ok.sum())

B, M = DATA / "bogota", DATA / "medellin"
def sel(path, key):
    return json.load(open(path))[key]

specs = [
 ("bogota_pm25_units", B/"bogota_pm25_pysr_units_pareto_front.csv", B/"bogota_master_dataset.parquet", "pm25", sel(B/"bogota_feature_selection.json","selected_no_lag")),
 ("bogota_pm25_freeform", B/"bogota_pysr_pareto_front.csv", B/"bogota_master_dataset.parquet", "pm25", sel(B/"bogota_feature_selection.json","selected_no_lag")),
 ("bogota_o3", B/"bogota_o3_pysr_pareto_front.csv", B/"bogota_o3_master_dataset.parquet", "o3", sel(B/"bogota_o3_feature_selection.json","selected_no_lag")),
 ("bogota_o3_falsification", B/"bogota_o3_temp_falsification_pareto.csv", B/"bogota_o3_master_dataset.parquet", "o3", sel(B/"bogota_o3_feature_selection.json","selected_no_lag")+["temp_2m"]),
 # 2026-10-06: the front the O3 lead equation now comes from -- units-constrained, temp_2m in the pool,
 # photochemical_potential split into no2 + radiation_mean (07_o3_units_with_temp_equation_discovery, seed 1)
 ("bogota_o3_units_temp", B/"bogota_o3_pysr_units_fire2_pm_pareto_front_seed0.csv", B/"bogota_o3_master_dataset.parquet", "o3",
  ["no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "rh_2m", "wind_speed_era5", "w700", "temp_2m", "pm25"]),   # 2026-10-20: lead front, PM2.5 and T offered
 ("bogota_o3_units_nopm", B/"bogota_o3_pysr_units_fire2_pareto_front_seed0.csv", B/"bogota_o3_master_dataset.parquet", "o3",
  ["no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "rh_2m", "wind_speed_era5", "w700", "temp_2m"]),
 ("bogota_o3_falsification_pm", B/"bogota_o3_temp_pm_falsification_pareto.csv", B/"bogota_o3_master_dataset.parquet", "o3",
  sel(B/"bogota_o3_feature_selection.json","selected_no_lag")+["temp_2m", "pm25"]),
 ("bogota_pm25_forecast", B/"bogota_pm25_forecast_interpretable_pysr_pareto.csv", None, "pm25", None),
 ("bogota_o3_forecast", B/"bogota_o3_forecast_interpretable_pysr_pareto.csv", None, "o3", None),
 ("medellin_pm25", M/"medellin_pysr_pareto_front.csv", M/"medellin_master_dataset.parquet", "pm25", sel(M/"medellin_feature_selection.json","selected_no_lag")),
 ("medellin_o3_refresh", M/"medellin_o3_pysr_pareto_front_refresh.csv", M/"medellin_o3_master_dataset.parquet", "o3", sel(M/"medellin_o3_feature_selection.json","selected_no_lag")),
]
from common.build_forecast_pool import build_forecast_pool
rows = []
for name, front_p, master_p, tgt, feats in specs:
    front = load_front(front_p)
    if master_p is None:   # forecast pools
        mp = B/("bogota_master_dataset.parquet" if tgt=="pm25" else "bogota_o3_master_dataset.parquet")
        m = pd.read_parquet(mp); m["date"] = pd.to_datetime(m["date"]); d = build_forecast_pool(m, targets=[tgt])
        jf = B/("bogota_forecast_feature_selection.json" if tgt=="pm25" else "bogota_o3_forecast_feature_selection.json")
        try: feats = json.load(open(jf))["no_self_lag_no_calendar"]["selected"]
        except KeyError: print("forecast selection not yet re-run for", name, "- skipped"); continue
    else:
        d = pd.read_parquet(master_p)
    d = d.dropna(subset=feats + [tgt]).reset_index(drop=True)
    for _, r in front.iterrows():
        r2, n = evaluate(r["equation"], d, tgt)
        rows.append(dict(front=name, complexity=int(r["complexity"]), r2_insample=r2, n=n, equation=r["equation"]))
out = pd.DataFrame(rows); out.to_csv(DATA_OUT/"front_r2_recomputed.csv", index=False)
print(out[out.front.isin(["bogota_pm25_units","bogota_o3_units_temp"]) & out.complexity.isin([13,15])][["front","complexity","r2_insample","n"]])
print(out.groupby("front").apply(lambda g: g.loc[g.r2_insample.idxmax(), ["complexity","r2_insample"]]).round(3))

Added fresh lag1/2/3 for 9 columns that didn't already have them: ['wind_dir', 'temp_10m', 'rh_10m', 'blh_min', 'blh_max', 'wind_speed_era5', 'wind_dir_era5', 'wind_dir_sin', 'wind_dir_cos']
Added fresh lag1/2/3 for 10 columns that didn't already have them: ['pm10', 'wind_dir', 'temp_10m', 'rh_10m', 'blh_min', 'blh_max', 'wind_speed_era5', 'wind_dir_era5', 'wind_dir_sin', 'wind_dir_cos']
                   front  complexity  r2_insample     n
10     bogota_pm25_units          13     0.683864  3283
11     bogota_pm25_units          15     0.686431  3283
80  bogota_o3_units_temp          13     0.587397  2917
82  bogota_o3_units_temp          15     0.609137  2917
                            complexity  r2_insample
front                                              
bogota_o3                           25        0.631
bogota_o3_falsification             25        0.639
bogota_o3_falsification_pm          25        0.639
bogota_o3_forecast                  25        0.477
bogota_o3_units_n

/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_40100/4146618397.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(out.groupby("front").apply(lambda g: g.loc[g.r2_insample.idxmax(), ["complexity","r2_insample"]]).round(3))
